# PerfectPetClub - Data Engineering ETL Pipeline
## Customer Churn Prediction & Analytics

### 1. Project Overview & Business Problem
**PerfectPetClub** is an online subscription service providing pet food and supplies. A key business challenge is **customer churn**—when customers disable the auto-renewal feature of their subscription (`auto_renew = 0`).

This ETL pipeline extracts data across relational and transactional sources, validates data quality, engineers predictive features for churn analysis, and exports the final curated dataset into an optimized Apache Parquet format.

### 2. Architecture: Extract - Transform - Load (ETL)
* **Extract:** Ingest customer profiles, subscriptions, sales, payments, and product reviews from SQLite (`PerfectPetClub.db`).
* **Transform:**
  1. *Data Quality Validation:* Run automated assertion checks (null handling, date order logic, identifier uniqueness, boundary checks).
  2. *Feature Engineering:* Construct the binary churn target (`is_churned`), subscription tenure, financial aggregations, and customer feedback metrics.
  3. *Data Ethics & GDPR:* Mask/drop personally identifiable information (PII) such as passwords and street addresses.
* **Load:** Write the clean customer-level feature dataset to `output/perfectpetclub_churn_features.parquet`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

sys.path.insert(0, str(Path("../src").resolve()))

from crm import extract_crm, transform_crm

In [9]:

# Call the function to extract our CRM dataset
crm_df = extract_crm("../data/PerfectPetClub_CRM_report.csv")

# Display the first five rows to confirm extraction worked
display(crm_df.head())




,customer_id,recieved_datatime,platform,type
0,55,2026-01-01 00:42:47.000,['e-mail'],complaint
1,49,2026-01-01 01:25:34.000,['live-chat'],query
2,44,2026-01-01 03:33:55.001,['telephone'],query
3,33,2026-01-01 05:42:16.002,['e-mail'],complaint
4,33,2026-01-01 06:25:03.002,['e-mail'],complaint


In [10]:
# Import transformation function
from crm import transform_crm
# Transform
cleaned_crm_df = transform_crm(crm_df)

# Display the first five rows of transformed data
display(cleaned_crm_df.head())


,customer_id,recieved_datatime,platform,type
0,55,2026-01-01 00:42:47.000,['e-mail'],complaint
1,49,2026-01-01 01:25:34.000,['live-chat'],query
2,44,2026-01-01 03:33:55.001,['telephone'],query
3,33,2026-01-01 05:42:16.002,['e-mail'],complaint
4,33,2026-01-01 06:25:03.002,['e-mail'],complaint


In [12]:
from extract_customer_subscriptions import extract_customer_subscriptions

df = extract_customer_subscriptions("../data/PerfectPetClub.db")
display(df.head())
raw_customers = extract_customer_subscriptions()

2026-11-10 15:32:33,043 - INFO - Connecting to database: ../data/PerfectPetClub.db
2026-11-10 15:32:33,090 - INFO - Customer count verified: 221 records.
2026-11-10 15:32:33,091 - INFO - Successfully extracted 221 customer records.


,customer_id,customer_name,email,county,postcode,subscription_id,sub_start_date,sub_end_date,auto_renew,subscription_tier,delivery_service
0,1,Anahit Sargsyan,Sargsyan701@somemail.co.uk,Berkshire,SL6 1CD,1,2026-01-01 00:00:00.000,2026-01-31 00:00:00.000,0,SuperSaver,Economy72
1,2,Leyla Mammadova,Mammadova282@example.com,Oxfordshire,OX7 5GH,2,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,SuperSaver,Economy72
2,3,Sana Shah,Sana858@notrealemail.com,Worcestershire,DY10 2JK,3,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,Premium,SignedFor24
3,4,Omar Farooq,Omar.Farooq4@example.com,Kent,TN9 1LM,4,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,Premium,SignedFor24
4,5,Alejandro Blanco,Alejandro357@somemail.co.uk,Hertfordshire,WD7 8PQ,5,2026-01-01 00:00:00.000,2026-05-01 00:00:00.000,0,SuperSaver,Economy72


2026-11-10 15:32:33,106 - INFO - Connecting to database: ../data/PerfectPetClub.db
2026-11-10 15:32:33,115 - INFO - Customer count verified: 221 records.
2026-11-10 15:32:33,115 - INFO - Successfully extracted 221 customer records.


In [18]:
# Test: Can our validation detect an invalid date?

import importlib
import extract_customer_subscriptions

# Reload the module to pick up changes
importlib.reload(extract_customer_subscriptions)

# Get the updated validation function
validate_customer_data = extract_customer_subscriptions.validate_customer_data

# Copy the original dataset
test_data = raw_customers.copy()

# Deliberately insert an invalid subscription date
test_data.loc[0, 'sub_end_date'] = 'invalid-date'

try:
    validate_customer_data(test_data)
    print("TEST FAILED: Invalid date was accepted.")

except AssertionError as e:
    if "subscription dates" in str(e).lower():
        print("TEST PASSED: Invalid date was detected.")
    else:
        print("TEST FAILED: An unrelated validation error occurred.")

    print(e)

2026-11-10 15:40:41,748 - INFO - Executing automated Data Quality checks on customer subscriptions...


TEST PASSED: Invalid date was detected.
DQ FAIL: Missing or invalid subscription dates.


## Stage 3: Transform — Feature Engineering & Integration
In this stage, we construct the predictive features for churn analysis:
* `is_churned`: **Target variable** (`1` if `auto_renew == 0` else `0`).
* `subscription_tenure_days`: Length of customer relationship in days.
* `avg_order_value_gbp`: Average transaction size.
* `one_star_reviews_count`: Dissatisfaction indicator.
* `postcode_district`: Area code for geographic segmentation.

In [22]:
import importlib
import extract_customer_subscriptions

# Reload latest changes
importlib.reload(extract_customer_subscriptions)

# Access transformation function
transform_customer_features = extract_customer_subscriptions.transform_customer_features

# Transform customer data
clean_customers = transform_customer_features(raw_customers)

clean_customers.head()

2026-11-10 15:44:27,457 - INFO - Applying feature engineering on customer subscriptions...
2026-11-10 15:44:27,519 - INFO - Target Engineered: 86/221 customers churned (38.91%).


,customer_id,customer_name,email,county,postcode,subscription_id,sub_start_date,sub_end_date,auto_renew,subscription_tier,delivery_service,is_churned,subscription_tenure_days,postcode_district
0,1,Anahit Sargsyan,Sargsyan701@somemail.co.uk,Berkshire,SL6 1CD,1,2026-01-01 00:00:00.000,2026-01-31 00:00:00.000,0,SuperSaver,Economy72,1,30,SL6
1,2,Leyla Mammadova,Mammadova282@example.com,Oxfordshire,OX7 5GH,2,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,SuperSaver,Economy72,0,330,OX7
2,3,Sana Shah,Sana858@notrealemail.com,Worcestershire,DY10 2JK,3,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,Premium,SignedFor24,0,330,DY10
3,4,Omar Farooq,Omar.Farooq4@example.com,Kent,TN9 1LM,4,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,Premium,SignedFor24,0,330,TN9
4,5,Alejandro Blanco,Alejandro357@somemail.co.uk,Hertfordshire,WD7 8PQ,5,2026-01-01 00:00:00.000,2026-05-01 00:00:00.000,0,SuperSaver,Economy72,1,120,WD7


### Feature Engineering Summary & Churn Analysis
Quick validation of churn distribution and key drivers:

In [23]:
import importlib
import extract_customer_subscriptions
# Churn rate by subscription tier
tier_summary = final_dataset.groupby('subscription_tier')['is_churned'].agg(
    total_customers='count',
    churn_rate='mean'
).reset_index()

tier_summary['churn_rate'] = (tier_summary['churn_rate'] * 100).round(1).astype(str) + '%'
print('--- Churn Rate by Subscription Tier ---')
print(tier_summary)

# Churned vs Retained spend & review comparison
print('\n--- Mean Metrics by Churn Status ---')
print(final_dataset.groupby('is_churned')[['total_spend_gbp', 'total_orders', 'avg_star_rating', 'one_star_reviews_count']].mean().round(2))

NameError: name 'final_dataset' is not defined

## Stage 4: Load — Export Curated Dataset to Apache Parquet
As specified in Task 2, the final transformed dataset must be exported as an Apache Parquet file.
Parquet provides columnar compression, preserves column data types, and is optimized for downstream analytical queries.

In [ ]:
# Create output directory
output_dir = Path('../output')
output_dir.mkdir(parents=True, exist_ok=True)

parquet_path = output_dir / 'perfectpetclub_churn_features.parquet'

# Export to Parquet
final_dataset.to_parquet(parquet_path, index=False, engine='pyarrow')

print(f'✓ Successfully loaded {len(final_dataset)} records into Parquet format:')
print(f'  Location: {parquet_path.resolve()}')
print(f'  File size: {parquet_path.stat().st_size:,} bytes')

In [ ]:
# Verification: Read back parquet file to guarantee integrity
df_verified = pd.read_parquet(parquet_path)
print(f'Parquet verification passed: {df_verified.shape[0]} rows, {df_verified.shape[1]} columns.')
print('\nDataset Columns:')
for col in df_verified.columns:
    print(f' - {col} ({df_verified[col].dtype})')

In [ ]:

# Final verification of feature engineering and Parquet output

# Check 1: Customer records have not been duplicated
assert final_dataset['customer_id'].is_unique, "Duplicate customers found"

# Check 2: Customer count remains unchanged
assert len(final_dataset) == len(raw_customers), "Customer count mismatch"

# Check 3: Churn calculation is correct
expected_churn = (final_dataset['auto_renew'] == 0).astype(int)
assert (final_dataset['is_churned'] == expected_churn).all(), "Incorrect churn values"

# Check 4: Subscription durations are not negative
assert (final_dataset['subscription_tenure_days'] >= 0).all(), "Negative duration found"

# Check 5: Parquet file contains the expected records and columns
assert df_verified.shape == final_dataset.shape, "Parquet dataset shape mismatch"
assert list(df_verified.columns) == list(final_dataset.columns), "Parquet columns mismatch"

print("PASS: Customer records are unique")
print("PASS: Customer count is correct")
print("PASS: Churn calculations are correct")
print("PASS: Subscription durations are valid")
print("PASS: Parquet structure is verified")
